In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep
import os
import random

In [2]:
# install on the control webiste librbay code.
# import subprocess
# import sys

# required_packages = ['python-docx',  'pypandoc']

# for package in required_packages:
#     try:
#         __import__(package)
#     except ImportError:
#         print(f'Installing {package}...')
#         subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

In [3]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'KW CBKU'  ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

# scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')  # if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running KW CBKU Web Scraping Tool v.1.1


In [4]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
# Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
         "download.prompt_for_download": False,
         "download.default_directory": tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1  # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

In [5]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------
# PAGE_SPECS: each entry defines one logical list with its URLs, list code, and display name.
# The root finance-companies page has no entity data; only its subpages are scraped.

PAGE_SPECS = [
    {
        'list_code': '1',
        'list_name': 'List of Banks',
        'urls': [
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/kuwaiti-banks/conventional-banks',
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/kuwaiti-banks/islamic-banks',
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/kuwaiti-banks/specialized-banks',
        ],
        'co_type': 'Bank',
    },
    {
        'list_code': '2',
        'list_name': 'List of Investment Companies - Conventional',
        'urls': [
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/investment-companies/conventional',
        ],
        'co_type': 'Investment Company',
    },
    {
        'list_code': '3',
        'list_name': 'List of Finance Companies',
        'urls': [
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/finance-companies/conventional-finance',
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/finance-companies/islamic-finance',
        ],
        'co_type': 'Finance Company',
    },
    {
        'list_code': '4',
        'list_name': 'List of Exchange Companies',
        'urls': [
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/exchange-companies',
        ],
        'co_type': 'Exchange Company',
    },
    {
        'list_code': '6',
        'list_name': 'List of Investment Companies - Islamic',
        'urls': [
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/investment-companies/islamic',
        ],
        'co_type': 'Investment Company',
    },
    {
        'list_code': '7',
        'list_name': 'List of Foreign Banks',
        'urls': [
            'https://www.cbk.gov.kw/en/supervision/regulated-entities/foreign-banks',
        ],
        'co_type': 'Foreign Bank',
    },
]

In [6]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def append_row(target, row):
    """Append one entity row; missing keys default to ''."""
    for k in target:
        target[k].append(row.get(k, ''))


_NORM_WS = re.compile(r'[\s\t\n\r\xa0]+')

def _normalize(text):
    """Collapse all whitespace (including non-breaking) to single spaces."""
    return _NORM_WS.sub(' ', text).strip()


LABEL_MAP = {
    'Address':        'Address_1',
    'Postal Address': 'Address_2',
    'Tel':            'Phone',
    'Fax':            'Fax',
    'Website':        'Website',
}

_ZIP_RE = re.compile(r',\s*(.+?)\s+(\d{5})\s*,')

def _extract_zip_city(postal_text):
    """Extract (city, zip) from 'P. O. Box: NN, AreaName NNNNN, State of Kuwait'."""
    m = _ZIP_RE.search(postal_text)
    if m:
        return m.group(1).strip(), m.group(2)
    return '', ''


def parse_cbk_h4_entities(soup):
    """Parse CBK regulated-entity pages.

    HTML structure per entity:
        <h4><span class="counter">N.</span>Entity Name</h4>
        <ul class="list-unstyled row">
            <li><span class="lbl">Label</span><p class="val">Value</p></li>
            ...
        </ul>

    Returns a list of dicts with keys:
        Name, Address_1, Address_2, Phone, Fax, Website
    """
    entities = []

    for h4 in soup.find_all('h4'):
        full_text = _normalize(h4.get_text())
        m = re.match(r'^\d+\.\s*(.+)$', full_text)
        if not m:
            continue

        name = m.group(1).strip()
        # strip trailing (*) annotation
        name = re.sub(r'\s*\(\*\)\s*$', '', name).strip()

        ent = {
            'Name': name,
            'Address_1': '',
            'Address_2': '',
            'Phone': '',
            'Fax': '',
            'Website': '',
        }

        # The <ul> with entity details follows the <h4> inside a wrapper div
        ul = h4.find_next_sibling('ul', class_='list-unstyled')
        if not ul:
            entities.append(ent)
            continue

        for li in ul.find_all('li', recursive=False):
            lbl_span = li.find('span', class_='lbl')
            val_p = li.find('p', class_='val')
            if not lbl_span or not val_p:
                continue

            label = _normalize(lbl_span.get_text())
            field = LABEL_MAP.get(label)
            if not field:
                continue

            if field == 'Website':
                link = val_p.find('a', href=True)
                if link:
                    ent['Website'] = link['href'].strip()
                else:
                    raw = _normalize(val_p.get_text())
                    if raw:
                        ent['Website'] = raw
            else:
                ent[field] = _normalize(val_p.get_text())

        entities.append(ent)

    return entities

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------
# CBK English pages: numbered entities in <h4> followed by label-value address blocks.

for spec_idx, spec in enumerate(PAGE_SPECS, start=1):
    list_key = f"{regulatorName} {spec['list_code']}"
    print(f"\n{'='*60}")
    print(f"[INFO] : Working {spec_idx}/{len(PAGE_SPECS)} - {list_key} ({spec['list_name']})")
    print(f"{'='*60}")

    spec_entity_count = 0

    for url in spec['urls']:
        print(f"  -> Fetching: {url}")

        drv = webdriver.Chrome(options=chromeOptions)
        drv.maximize_window()
        drv.get(url)
        sleep(random.uniform(4, 8))

        try:
            show_all = drv.find_element(By.CSS_SELECTOR,
                'nav.pagination-wrapper a.page-link[href*="showAll=yes"]')
            show_all.click()
            print(f"  [INFO] Clicked 'Show All' (page was paginated)")
            sleep(random.uniform(3, 6))
        except Exception:
            pass

        soup = BeautifulSoup(drv.page_source, 'html.parser')
        entities = parse_cbk_h4_entities(soup)

        if not entities:
            print(f"  [WARN] No numbered h4 entities found at {url}")

        for ent in entities:
            city, zipcode = _extract_zip_city(ent['Address_2'])
            append_row(sqldict, {
                'Name':            ent['Name'],
                'Address_1':       ent['Address_2'],
                'City':            city,
                'Zip':             zipcode,
                'Phone':           ent['Phone'],
                'Fax':             ent['Fax'],
                'Website':         ent['Website'],
                'Typology':          spec['co_type'],
                'Cntry':           'KW',
                'ListProcessDate': processdate,
                'RegCtry':         'KW',
                'RegCode':         'CBKU',
                'ListCode':        spec['list_code'],
                'RegulationType':  'Regulated',
                'ListName':        spec['list_name'],
            })
            sqldict = bourange_same_length_array(sqldict)

        spec_entity_count += len(entities)
        print(f"  [INFO] {len(entities)} entities from this URL")

        drv.quit()
        sleep(random.uniform(2, 5))

    print(f"[INFO] : Total {spec_entity_count} entities for {list_key}")
    print(f"[INFO] : Completed {list_key}")


[INFO] : Working 1/6 - KW CBKU 1 (List of Banks)
  -> Fetching: https://www.cbk.gov.kw/en/supervision/regulated-entities/kuwaiti-banks/conventional-banks
  [INFO] 5 entities from this URL
  -> Fetching: https://www.cbk.gov.kw/en/supervision/regulated-entities/kuwaiti-banks/islamic-banks
  [INFO] 4 entities from this URL
  -> Fetching: https://www.cbk.gov.kw/en/supervision/regulated-entities/kuwaiti-banks/specialized-banks
  [INFO] 1 entities from this URL
[INFO] : Total 10 entities for KW CBKU 1
[INFO] : Completed KW CBKU 1

[INFO] : Working 2/6 - KW CBKU 2 (List of Investment Companies - Conventional)
  -> Fetching: https://www.cbk.gov.kw/en/supervision/regulated-entities/investment-companies/conventional
  [INFO] 10 entities from this URL
[INFO] : Total 10 entities for KW CBKU 2
[INFO] : Completed KW CBKU 2

[INFO] : Working 3/6 - KW CBKU 3 (List of Finance Companies)
  -> Fetching: https://www.cbk.gov.kw/en/supervision/regulated-entities/finance-companies/conventional-finance
  [IN

In [8]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, sheet_name='SQL Ready', index=False)
try:
    driver.quit()
except Exception:
    pass
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully")

[INFO] : Excel file 'KW CBKU SQL Ready 2026-04-20 16.08.25.xlsx' saved successfully


In [9]:
#------------------------------------------------ Data Integrity & Consistency Check ----------------------------------------

print('=' * 80)
print('DATA INTEGRITY & CONSISTENCY VERIFICATION')
print('=' * 80)

# Derive expected lists from PAGE_SPECS (single source of truth)
expected_lists = {spec['list_code']: spec['list_name'] for spec in PAGE_SPECS}

print('\n1. DATAFRAME SHAPE:')
print(f'   Total rows collected: {len(df)}')
print(f'   Total columns: {len(df.columns)}')

print('\n2. DATA DISTRIBUTION BY LIST:')
list_summary = df.groupby('ListCode').agg({
    'Name': 'count',
    'ListName': 'first',
    'RegCtry': 'first',
    'RegCode': 'first'
}).rename(columns={'Name': 'Count'})
print(list_summary)

print('\n3. NULL VALUES CHECK (Data Completeness):')
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print('   PASS: No null values found')
else:
    print('   FAIL: Null values detected in:')
    for col, count in null_counts[null_counts > 0].items():
        print(f'      - {col}: {count} nulls')

print('\n4. CONSISTENCY WITH PAGE_SPECS / README:')
print('\n   List | Expected Name                             | Actual Name                             | Count | Status')
print('   -----|---------------------------------------------|----------------------------------------|-------|--------')
for list_code, expected_name in expected_lists.items():
    data = df[df['ListCode'] == list_code]
    if len(data) == 0:
        print(f"   {list_code:<4} | {expected_name:<42} | (NO DATA)                              |   0   | MISSING")
    else:
        actual_name = data['ListName'].iloc[0]
        count = len(data)
        match = 'OK' if expected_name.lower() == actual_name.lower() else 'MISMATCH'
        print(f'   {list_code:<4} | {expected_name:<42} | {actual_name:<39} | {count:>5} | {match}')

print('\n5. REGCTRY & REGCODE VALIDATION:')
regctry_values = df['RegCtry'].unique()
regcode_values = df['RegCode'].unique()
regctry_status = 'CORRECT' if all(v == 'KW' for v in regctry_values) else 'INCORRECT'
regcode_status = 'CORRECT' if all(v == 'CBKU' for v in regcode_values) else 'INCORRECT'
print(f'   RegCtry values: {regctry_values} {regctry_status}')
print(f'   RegCode values: {regcode_values} {regcode_status}')

print('\n6. KEY FIELDS VALIDATION:')
key_fields = ['Name', 'Phone', 'Fax', 'CoType', 'Address_1', 'City', 'Zip', 'Website']
for field in key_fields:
    non_empty = (df[field].astype(str).str.strip() != '').sum()
    total = len(df)
    status = 'OK' if non_empty == total else f'{non_empty}/{total} filled'
    print(f'   {field:<12}: {status}')
print(f"   ListProcessDate: {df['ListProcessDate'].iloc[0] if len(df) > 0 else 'N/A'}")

print('\n7. SAMPLE DATA (first 5 rows):')
if len(df) > 0:
    print(df[['Name', 'CoType', 'Address_1', 'City', 'Zip', 'Phone', 'Fax', 'Website', 'ListCode']].head(5).to_string())
else:
    print('   NO DATA COLLECTED')

print('\n8. LIST COVERAGE vs PAGE_SPECS:')
print('   Code | Expected Name                             | Rows | Status')
print('   -----|---------------------------------------------|------|----------')
total_expected = len(expected_lists)
total_collected = 0
for list_code, expected_name in expected_lists.items():
    count = len(df[df['ListCode'] == list_code])
    total_collected += (1 if count > 0 else 0)
    status = 'COLLECTED' if count > 0 else 'MISSING'
    print(f"   {list_code:<4} | {expected_name:<42} | {count:>4} | {status}")

print(f'\n   Lists collected: {total_collected}/{total_expected}')

print('\n' + '=' * 80)
print('SUMMARY:')
print('=' * 80)
print(f'Total rows in DataFrame: {len(df)}')
print(f'Expected lists coverage: {total_collected}/{total_expected}')
missing_lists = total_expected - total_collected
if missing_lists > 0:
    print(f'Missing lists count: {missing_lists}')
    print('  ACTION: Re-run scraping cell; if still empty, inspect CBK page HTML and adjust parse_cbk_h4_entities.')
else:
    print('All lists collected successfully')

print('=' * 80)

DATA INTEGRITY & CONSISTENCY VERIFICATION

1. DATAFRAME SHAPE:
   Total rows collected: 50
   Total columns: 44

2. DATA DISTRIBUTION BY LIST:
          Count                                     ListName RegCtry RegCode
ListCode                                                                    
1            10                                List of Banks      KW    CBKU
2            10  List of Investment Companies - Conventional      KW    CBKU
3             2                    List of Finance Companies      KW    CBKU
4            10                   List of Exchange Companies      KW    CBKU
6             8       List of Investment Companies - Islamic      KW    CBKU
7            10                        List of Foreign Banks      KW    CBKU

3. NULL VALUES CHECK (Data Completeness):
   PASS: No null values found

4. CONSISTENCY WITH PAGE_SPECS / README:

   List | Expected Name                             | Actual Name                             | Count | Status
   -----|-----